# Python System Environment — The Raise of Python

*A young developer, an old-fashioned administrator, and one shared Ubuntu server.*

## Introduction — Alice Meets Bob

> **Bob:** Welcome to my server. One machine, one Python. I've managed it this way for years.
>
> **Alice:** I'm Alice, a developer on the new project. Before we install anything, let's find out what Ubuntu actually manages.
>
> **Bob:** Must we? I already know where the executable is.
>
> **Alice:** Knowing the executable is a start. Its libraries and package directories matter too.

Run the cells in order in the [lab Dev Container](./README.md). The image installs Python and Jupyter; the commands below run inside the disposable Ubuntu 24.04 container as Bob. The technical companion is [Python System Environment](../../docs/chapter-01/section-01.md).

## Python on Ubuntu

> **Alice:** Ubuntu provides the `python3` package through APT. The image was built with `apt-get install python3 python3-dev python3-pip python3-venv`. Let's see the installed packages and the interpreter.
>
> **Bob:** So `/usr/bin/python3` belongs to the distribution, not just to my application?
>
> **Alice:** Exactly. OS tools may depend on it. If you need another version, install it side by side and select it explicitly instead of replacing that executable.

In [ ]:
%%bash
set -eu
. /etc/os-release
printf '%s\n' "$PRETTY_NAME"
command -v python3
/usr/bin/python3 --version
dpkg-query -W python3 python3-dev python3-pip python3-venv
readlink -f /usr/bin/python3
dpkg -S /usr/bin/python3


## Installation Footprint and Search Paths

> **Bob:** There's more than one Python directory? That seems untidy.
>
> **Alice:** Each has a purpose: the standard library, distribution packages, local administrator packages, user packages, and headers. `PATH` selects a command; Python uses `sys.path` to resolve imports. Those are different search lists.

Inspect the interpreter's actual configuration rather than assuming a minor version or path.

In [ ]:
%%bash
set -eu
printf 'PATH=%s\n' "$PATH"
/usr/bin/python3 -m site
/usr/bin/python3 -c 'import sys, sysconfig; print("executable:", sys.executable); print("stdlib:", sysconfig.get_path("stdlib")); print("headers:", sysconfig.get_path("include")); print("sys.path:"); print(*sys.path, sep="\n")'


## The Distribution Package Target

> **Bob:** I install everything system-wide. That's what APT is for.
>
> **Alice:** APT is appropriate for distribution-managed packages and their OS dependencies. Let's install `python3-psutil`, then ask both Python and APT where it lives.

The package installation changes only this container and requires network access. The image supplies passwordless sudo for Bob.

In [ ]:
%%bash
set -eu
sudo apt-get update -qq
sudo apt-get install -y python3-psutil
/usr/bin/python3 -c 'import psutil; print("psutil:", psutil.__file__); assert psutil.__file__.startswith("/usr/lib/python3/dist-packages/")'
dpkg -L python3-psutil | grep '/dist-packages/psutil/__init__.py'


## User and Administrator Targets

> **Bob:** If Alice needs a package, I'll just use pip on the system interpreter.
>
> **Alice:** Ubuntu marks that interpreter externally managed (PEP 668). The override `--break-system-packages` is explicit because it bypasses that protection. Application dependencies should normally go in a virtual environment. Our Jupyter tooling lives in its own environment.
>
> **Bob:** What about a user install?
>
> **Alice:** It lives under the user's home, but is still shared by that user's projects. Let's inspect an isolated user target as Alice.

The override is needed even for `--user` on Ubuntu 24.04. Install a small pinned package as Alice inside the container, then compare visibility for Alice and Bob.

In [ ]:
%%bash
set -eu
sudo -H -u alice /usr/bin/python3 -m pip install --user --break-system-packages 'idna==3.10'
sudo -H -u alice /usr/bin/python3 -c 'import idna, site; print("Alice user site:", site.getusersitepackages()); print("Alice idna:", idna.__file__); assert idna.__file__.startswith(site.getusersitepackages())'
/usr/bin/python3 -c 'import importlib.util, site; print("Bob user site:", site.getusersitepackages()); spec = importlib.util.find_spec("idna"); print("Bob idna:", spec); assert spec is None'


## Epilogue — One Interpreter, Many Projects

> **Bob:** Alice's user package is separate from my account, but her two projects still share it. A version upgrade could break one of them.
>
> **Alice:** Right. The system interpreter and standard library are shared, APT owns distribution packages, and user installs are per user, not per project. Let's give each project its own environment.

Continue with [Python `venv` Environments — The Isolation Awakens](../python_venv_lab/python_venv_lab.ipynb).